In [2]:
pip install easyocr pymupdf pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 10.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 11.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 10.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 11.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.2 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 971.0/971.0 kB 8.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 11.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [easyocr]2/13 [easyocr]ion]]-headless]
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain langchain-community
%pip install -Uq python_dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [16]:
import json
from typing import List
from transformers import pipeline
from PIL import Image
import base64
import io
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Unstructured for Document Parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings                
from langchain_community.vectorstores import Chroma

In [20]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="fast", # Method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        # extract_image_block_types=["Image"], # Grab images found in the PDF
        # extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "./docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path)

📄 Partitioning document: ./docs/attention-is-all-you-need.pdf
✅ Extracted 393 elements


In [21]:
elements

In [22]:
# All unique types of elements extracted
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [23]:
elements[30].to_dict()

{'type': 'Title',
 'element_id': '36dba529fb5f1daa7dfa890bb9cac514',
 'text': '3 Model Architecture',
 'metadata': {'coordinates': {'points': ((108.0, 642.2234368),
    (108.0, 654.1786368),
    (226.0934656, 654.1786368),
    (226.0934656, 642.2234368)),
   'system': 'PixelSpace',
   'layout_width': 612.0,
   'layout_height': 792.0},
  'file_directory': './docs',
  'filename': 'attention-is-all-you-need.pdf',
  'last_modified': '2026-05-24T12:35:04',
  'page_number': 2,
  'languages': ['eng'],
  'filetype': 'application/pdf'}}

In [24]:
# Gather all titles
titles = [element for element in elements if element.category == 'Title']       # For other types, just change the category name
print(f"Found {len(titles)} titles")

# Get to know about a title chosen at random
titles[0].to_dict()

Found 158 titles


{'type': 'Title',
 'element_id': '35f26475b2e7ba32ac5400036e6361d5',
 'text': '] L C . s c [',
 'metadata': {'coordinates': {'points': ((16.34, 318.35999999999996),
    (16.34, 378.9),
    (36.34, 378.9),
    (36.34, 318.35999999999996)),
   'system': 'PixelSpace',
   'layout_width': 612.0,
   'layout_height': 792.0},
  'file_directory': './docs',
  'filename': 'attention-is-all-you-need.pdf',
  'last_modified': '2026-05-24T12:35:04',
  'page_number': 1,
  'languages': ['eng'],
  'filetype': 'application/pdf'}}

In [25]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 34 chunks


In [26]:
# View all chunks
# chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>"}

In [27]:
# View a single chunk
# chunks[2].to_dict()

# View original elements
chunks[1].metadata.orig_elements[1].to_dict()

{'type': 'Title',
 'element_id': '9a891d4dd279cc15d3ce9ae53cb91e8b',
 'text': 'Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com',
 'metadata': {'coordinates': {'points': ((364.84899999999993, 283.4561172),
    (364.84899999999993, 316.7216972),
    (485.1145184199999, 316.7216972),
    (485.1145184199999, 283.4561172)),
   'system': 'PixelSpace',
   'layout_width': 612.0,
   'layout_height': 792.0},
  'file_directory': './docs',
  'filename': 'attention-is-all-you-need.pdf',
  'last_modified': '2026-05-24T12:35:04',
  'page_number': 1,
  'languages': ['eng'],
  'filetype': 'application/pdf'}}

In [31]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data




model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

model.eval()




def generate_response(prompt, max_new_tokens=200):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return generated_text[len(prompt):].strip()




def create_ai_enhanced_summary(
    text: str,
    tables: list[str],
    images: list[str]
) -> str:

    try:

        prompt = f"""
You are creating a searchable document summary.

TEXT CONTENT:
{text}

"""

        if tables:

            prompt += "\nTABLE CONTENT:\n"

            for i, table in enumerate(tables):

                prompt += f"\nTable {i+1}:\n{table}\n"

        prompt += """

Generate:
1. Key facts
2. Important numbers
3. Topics discussed
4. Searchable keywords
5. Questions this could answer

SEARCHABLE SUMMARY:
"""

        summary = generate_response(
            prompt,
            max_new_tokens=200
        )

        return summary

    except Exception as e:

        print(f"❌ AI summary failed: {e}")

        summary = text[:300]

        if tables:
            summary += f"\n[Contains {len(tables)} table(s)]"

        return summary




def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}")
        
        # Create AI-enhanced summary if chunk has tables
        if content_data['tables']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5520.74it/s]


🧠 Processing chunks with AI Summaries...
   Processing chunk 1/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 2/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 3/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 4/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 5/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 6/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 7/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 8/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 9/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no ta

In [32]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "3 2 0 2\\n\\ng u A 2\\n\\n] L C . s c [\\n\\n7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a\\n\\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\\n\\nAttention Is All You Need\\n\\nAshish Vaswani\\u2217 Google Brain avaswani@google.com\\n\\nNoam Shazeer\\u2217 Google Brain noam@google.com\\n\\nNiki Parmar\\u2217 Google Research nikip@google.com\\n\\nJakob Uszkoreit\\u2217 Google Research usz@google.com\\n\\nLlion Jones\\u2217 Google Research llion@google.com", "tables_html": []}'}, page_content='3 2 0 2\n\ng u A 2\n\n] L C . s c [\n\n7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a\n\nProvided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.\n\nAttention Is All You Need\n\nAshish Vaswani∗ Google Brain avaswani@google.

In [33]:
# Optional: This is only for viewing the enhanced content and metadata in a clean format, not necessary for RAG itself

def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 34 chunks to chunks_export.json


In [34]:
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")
        
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...


/tmp/ipykernel_4018585/69613739.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6213.43it/s]


--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [35]:
# After your retrieval
query = "What are the two main components of the Transformer architecture? "
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 3 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': '6.2 Model Variations\n\nTo evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the\n\n5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectively.\n\n8\n\nTable 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Listed perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to per-word perplexities.',
  'metadata': {'original_content': {'raw_text': '6.2 Model Variations\n\nTo evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the\n\n5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for

In [38]:
# Complete Pipeline

def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db


db = run_complete_ingestion_pipeline("./docs/attention-is-all-you-need.pdf")

🚀 Starting RAG Ingestion Pipeline
📄 Partitioning document: ./docs/attention-is-all-you-need.pdf
✅ Extracted 393 elements
🔨 Creating smart chunks...
✅ Created 34 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 2/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 3/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 4/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 5/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 6/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 7/34
     Types found: ['text']
     Tables: 0
     → Using raw text (no tables/images)
   Processing chunk 8/34
     Types

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6421.51it/s]


--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv2/chroma_db
🎉 Pipeline completed successfully!


In [39]:
query = "How many attention heads does the Transformer use, and what is the dimension of each head?"

retriever = db.as_retriever(search_kwargs={"k": 3})

chunks = retriever.invoke(query)


def generate_final_answer(chunks, query):

    """
    Generate final answer using retrieved chunks
    with TinyLlama instead of OpenAI
    """

    try:

        prompt = f"""
            You are a helpful AI assistant.

            Answer the user's question ONLY using the provided documents.

            If the answer is not found in the documents, say:
            "I don't have enough information to answer that question based on the provided documents."

            USER QUESTION:
            {query}

            DOCUMENTS:
            """


        for i, chunk in enumerate(chunks):

            prompt += f"\n--- DOCUMENT {i+1} ---\n"

            # Use original raw content if available
            if "original_content" in chunk.metadata:

                original_data = json.loads(
                    chunk.metadata["original_content"]
                )

                # Raw text
                raw_text = original_data.get(
                    "raw_text",
                    ""
                )

                if raw_text:

                    prompt += f"\nTEXT:\n{raw_text}\n"

                # Tables
                tables_html = original_data.get(
                    "tables_html",
                    []
                )

                if tables_html:

                    prompt += "\nTABLES:\n"

                    for j, table in enumerate(tables_html):

                        prompt += (
                            f"\nTable {j+1}:\n{table}\n"
                        )

            else:

                # Fallback
                prompt += f"\n{chunk.page_content}\n"


        prompt += """

            Provide:
            1. A clear answer
            2. Important technical details
            3. Relevant numbers if available

            ANSWER:
            """
        

        answer = generate_response(
            prompt,
            max_new_tokens=250
        )


        if not answer.strip():

            answer = (
                "I don't have enough information "
                "to answer that question based on "
                "the provided documents."
            )

        return answer

    except Exception as e:

        print(f"❌ Answer generation failed: {e}")

        return (
            "Sorry, I encountered an error "
            "while generating the answer."
        )


final_answer = generate_final_answer(
    chunks,
    query
)

print("\nFINAL ANSWER:\n")
print(final_answer)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL ANSWER:

ically implemented using a multi-head attention mechanism, where each head computes a weighted sum of the output of its corresponding key-value pairs.

In the Transformer, the attention mechanism is implemented using a multi-head self-attention mechanism. The self-attention mechanism takes as input a query and a set of key-value pairs, and computes a weighted sum of the output of the key-value pairs. The output is then used to update the query and key vectors.

In the Transformer, the self-attention mechanism is implemented using
